In [ ]:
import pandas as pd

In [ ]:
trxn= pd.read_csv('Downloads/Retail_Data_Transactions.csv')
trxn

In [ ]:
response=pd.read_csv('Downloads/Retail_Data_Response.csv')
response

In [ ]:
df= trxn.merge(response, on='customer_id', how='right')
df

In [ ]:
#features
df.dtypes

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna()
df

In [ ]:
#change dtype
set(df['response'])
df['trans_date']= pd.to_datetime(df['trans_date'])
df['response']=df['response'].fillna(0).astype('int64')
df

In [ ]:
set(df['response'])

In [ ]:
df.dtypes


In [ ]:
#z score
from scipy import stats
import numpy as np

#calc z score
z_scores=np.abs(stats.zscore(df['tran_amount']))

#set a threshold
threshold= 3
outliers=z_scores>threshold 

print(df['tran_amount'][outliers])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x=df['response'])
plt.show()

In [ ]:
sns.boxplot(x=df['tran_amount'])
plt.show()

In [ ]:
#creating new column
df['month']=df['trans_date'].dt.month
df

In [ ]:
#which 3 month have had the highest number of transactio amount???
Monthly_sales=df.groupby('month')['tran_amount'].sum()
Monthly_sales=Monthly_sales.sort_values(ascending=False).reset_index().head(3)
Monthly_sales


In [ ]:
#Customers having highest numbers of order
customer_counts=df['customer_id'].value_counts().reset_index()
customer_counts

In [ ]:
top_5_cust=customer_counts.sort_values(by='count',ascending=False).head(5)
top_5_cust

In [ ]:
sns.barplot(x='customer_id',y='count',data=top_5_cust)

In [ ]:
customer_sales=df.groupby('customer_id')['tran_amount'].sum().reset_index()
customer_sales
#sort
top_5_sale=customer_sales.sort_values(by='tran_amount',ascending=False).head(5)
top_5_sale

In [ ]:
sns.barplot(x='customer_id',y='tran_amount',data=top_5_sale)


# Advance analysis #

## Time series ##

In [ ]:
import matplotlib.dates as mdates
df['month_year']=df['trans_date'].dt.to_period('M')
df

In [ ]:
monthly_sales=df.groupby('month_year')['tran_amount'].sum()
monthly_sales.index = monthly_sales.index.to_timestamp()

plt.figure(figsize=(12,6))
plt.plot(monthly_sales.index ,monthly_sales.values)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xlabel('Month-year')
plt.ylabel('Sales')
plt.title('Mothly sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Cohort segmentation ##

In [ ]:
#recency
recency=df.groupby('customer_id')['trans_date'].max()

#frequency
frequency=df.groupby('customer_id')['trans_date'].count()

#monetary
monetary=df.groupby('customer_id')['tran_amount'].sum()

#Combine
rfm=pd.DataFrame({'recency':recency ,'frequency':frequency, 'monetary':monetary})
rfm

In [ ]:
#customer_segmentation
def segment_customer(row):
    if row['recency'].year>=2012 and row['frequency']>=15 and row['monetary']>=1000:
        return 'P0'
    elif (2011<=row['recency'].year) and (row['recency'].year<2012) and (10<row['frequency']<=15) and (500<row['monetary']<=1000):
        return 'P1'
    else:
        return 'P2'
rfm['segment']=rfm.apply(segment_customer, axis=1)
rfm

## Churn analysis ##

In [ ]:
#Count the numbers of churn and active customers
churn_count=df['response'].value_counts()
#Plot
churn_count.plot(kind='bar')

In [ ]:
top_3_cus=monetary.sort_values(ascending=False).head(3).index

top_customer_df=df[df['customer_id'].isin(top_3_cus)]

top_customer_sales= top_customer_df.groupby(['customer_id','month_year'])['tran_amount'].sum().unstack(level=0)
top_customer_sales.plot(kind='line')